In [0]:
# !pip install autoviz
!pip install lightgbm shap

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col
# from autoviz.AutoViz_Class import AutoViz_Class


In [0]:
def load_and_clean_credit_card_data():
    """
    Load credit card history data and apply business exclusion filters.

    Steps:
      1. Query the credit card history view, filtering to the first 24 months
         after approval for accounts approved between Jul 2022 and Jun 2025.
      2. Compute a per-card count of OPEN snapshots (needed for Exclusion 9).
      3. Build a composite exclusion condition and remove matching rows.

    Returns:
        tuple[DataFrame, DataFrame]:
            df       – raw data with the 'open_count' helper column.
            df_clean – filtered data with exclusions applied and 'open_count' dropped.
    """

   # --- Step 1: Load raw data within the 24-month observation window ----------

    df = spark.sql("""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, 12)
          AND APPROVEDATE > '2022-06-30'
          AND APPROVEDATE < '2025-07-01'
    """)

    # --- Step 2: Count OPEN snapshots per card (for Exclusion 9) ---------------
    w = Window.partitionBy("CREDITCARDNO")
    df = df.withColumn(
        "open_count",
        F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
    )

    # --- Step 3: Define exclusion conditions -----------------------------------
    exclude_cond = (
        (F.upper(F.trim(F.col("CLASSCODE"))) == "SME") |                                # Exclusion 1: SME class code
        (~F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS")) |              # Exclusion 2: Non-CARDS segment
        (F.col("VIP") == 1) |                                                          # Exclusion 3a: VIP customers
        (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |                         # Exclusion 3b: Ruling family
     #  (F.col("AECB_DATA").isNull()) |                                                 # Exclusion 4: No AECB data (disabled)
        (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) != "SALARIED") |                 # Exclusion 6: Non-salaried
        (F.col("PRIMARYCARD") == 0) |                                                  # Exclusion 7: Supplementary cards
        (F.upper(F.trim(F.col("CLASSCODE"))) == "STAFF") |                              # Exclusion 8: Staff cards
        (F.col("open_count") < 3) | # Exclusion 9: Fewer than 3 OPEN snapshots
        (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM") |
        (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin("PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))  # Exclusion 10: Unwanted segment policies
    )

    # --- Step 4: Apply exclusions and drop helper column -----------------------
    df_clean = df.filter(~exclude_cond).drop("open_count")

    return df, df_clean


# Execute and inspect
df, df_clean = load_and_clean_credit_card_data()
df_pd = df_clean.toPandas()
print(df_clean.columns)

In [0]:
cols = ['SNAPSHOT_DATE','CRNNO','CREDITCARDNO','CARDCREATEDATE','APPROVEDATE','DELINQUENCYBUCKET']

def filter_cards_with_sufficient_open_snapshots(df_pd, min_open=3):
    """
    Filter credit cards to retain only those with a sufficient number of
    "OPEN" status snapshots after all prior exclusions have been applied.

    ---------------------------------------------------------------------------
    Context:
      During the initial data extract (Exclusion 9), we already removed cards
      that had fewer than 3 OPEN snapshots overall. However, after applying the
      remaining filters (e.g. Exclusion 6: non-salaried), some cards that
      originally had ≥ 3 OPEN snapshots now fall below that threshold in the
      cleaned dataset.

    Why this happens:
      These are cases where a customer's EMPLOYMENT_CATEGORY changed mid-
      observation (e.g. from "SALARIED" to "SELF-EMPLOYED"). The non-salaried
      snapshots were excluded, leaving fewer than 3 OPEN & salaried records.

    Why we treat them separately / remove them:
      A change in employment status signals a behavioural shift — the customer
      may no longer have a stable, predictable income stream, which materially
      affects credit-card repayment patterns. Mixing these customers with the
      stable-salaried population would introduce noise into the model.

    Volume check:
      This affects only ~300 cards, so removing them has negligible impact on
      overall sample size while improving data quality.
    ---------------------------------------------------------------------------

    Args:
        df_pd (pd.DataFrame): Cleaned credit card DataFrame (post-exclusion)
                              containing 'CREDITCARDNO' and 'CURRENT_STATUS' columns.
        min_open (int): Minimum number of "1.OPEN" snapshots a card must have
                        to be retained. Default is 3.

    Returns:
        pd.DataFrame: Filtered DataFrame containing only cards with at least
                      `min_open` OPEN snapshots.
    """

    # Step 1: Count the number of snapshots per status for each credit card
    open_counts = (
        df_pd
        .groupby(['CREDITCARDNO'])["CURRENT_STATUS"]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Step 2: Identify cards that meet the minimum OPEN snapshot threshold
    creditcards_more_than_2_open = open_counts.loc[
        open_counts["1.OPEN"] >= min_open, "CREDITCARDNO"
    ]

    # Step 3: Filter the original DataFrame to keep only qualifying cards
    df_clean_pd = df_pd.loc[
        df_pd['CREDITCARDNO'].isin(creditcards_more_than_2_open.to_list())
    ]

    print(f"Cards before filter : {df_pd['CREDITCARDNO'].nunique()}")
    print(f"Cards after filter  : {df_clean_pd['CREDITCARDNO'].nunique()}")
    print(f"Cards removed       : {df_pd['CREDITCARDNO'].nunique() - df_clean_pd['CREDITCARDNO'].nunique()}")
    print(f"Rows before: {df_pd.shape[0]}  |  Rows after: {df_clean_pd.shape[0]}")

    return df_clean_pd


# Execute: retain only cards with ≥ 3 OPEN snapshots
df_clean_pd = filter_cards_with_sufficient_open_snapshots(df_pd, min_open=3)
df_clean_pd.shape

In [0]:
print(df_clean_pd.shape)

# Save to Parquet in Unity Catalog Volumes
df_clean_pd.to_parquet("/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/input/df_clean.parquet", index=False)

In [0]:
# df_pd.CUSTOMER_SEGMENT.value_counts()
# SALARIED                  1183386
# WEALTH MANAGEMENT CXMS       8019
# SELF EMPLOYED                 165
# SECURED CARDS                  12
# df_pd.CLASSCODE.unique()
# ['UAE', 'UAE   ', 'EXP', 'EXP   ', 'AECB1', 'AECB6', 'AECB4',
    #    'AECB12', 'NRE', 'ALOC', 'AECB5', 'AECB3', 'AECB1 ']
# df_pd.F_SEGMENT.unique()
# ['Salaried', 'Secured/WM', 'CHECK', 'SELF EMPLOYED']
df_pd.F_SEGMENT_NEW.unique()

In [0]:
# --- Target Variable Definition ---
# 1. Filter snapshots within 12 months of approval date
# 2. Group by CREDITCARDNO and take max DELINQUENCYBUCKET in that window
# 3. Define target: bad (1) if max bucket > 3, good (0) if max bucket == 0, indeterminate (NaN) otherwise

df_target = (
    df_clean_pd[df_clean_pd['SNAPSHOT_DATE'] <= df_clean_pd['APPROVEDATE'] + pd.DateOffset(months=12)]
    .sort_values(['CREDITCARDNO', 'SNAPSHOT_DATE'])
    .groupby('CREDITCARDNO', as_index=False)
    .agg(
        max_delinquency=('DELINQUENCYBUCKET', 'max'),
        approvedate=('APPROVEDATE', 'first')
    )
)

# Convert to numeric in case it's stored as string
df_target['max_delinquency'] = pd.to_numeric(df_target['max_delinquency'], errors='coerce')

# Final target: 1 = Bad (max bucket > 3), 0 = Good (max bucket == 0), NaN = Indeterminate
df_target['target'] = np.where(
    df_target['max_delinquency'] > 3, 1,
    np.where(df_target['max_delinquency'] == 0, 0, np.nan)
)

print(f"Total cards: {len(df_target)}")
print(f"\nTarget distribution:")
print(df_target['target'].value_counts(dropna=False))
print(f"\nBad rate: {df_target['target'].mean():.4%} (excluding indeterminate)")
df_target.head(10)

In [0]:
# Merge target labels with cleaned credit card data
df_merged = df_clean_pd.merge(df_target[['CREDITCARDNO', 'max_delinquency', 'target']], on='CREDITCARDNO', how='inner')

# Remove indeterminate cases (target = NaN), keep only 0 and 1
df_merged = df_merged[df_merged['target'].isin([0, 1])].reset_index(drop=True)

print(f"Shape after merge: {df_merged.shape}")
print(f"Unique cards: {df_merged['CREDITCARDNO'].nunique()}")
print(f"\nTarget distribution:")
print(df_merged['target'].value_counts(dropna=False))
df_merged.head()

In [0]:
# Aggregate to one row per customer: take the worst (max) target label
df_ref = (
    df_merged
    .groupby('REFERENCE_NUMBER', as_index=False)
    .agg(target=('target', 'max'))
)

print(f"Unique customers: {df_ref.shape[0]}")
print(f"\nTarget distribution:")
print(df_ref['target'].value_counts(dropna=False))
print(f"\nBad rate: {df_ref['target'].mean():.4%}")
df_ref.head()

In [0]:
# Retrieve all column names for the dim_customer table
columns_df = spark.sql("DESCRIBE mis_automation.silver_fin_db.dim_customer")
columns_df.display()

In [0]:

cif_list = list(df_merged.CIF.unique())
spark.createDataFrame([(c,) for c in cif_list], ["CIF"]).createOrReplaceTempView("cif_filter")

# Avoid ArrowInvalid for out-of-range timestamps by disabling Arrow optimization
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

demo = spark.sql("""
    SELECT d.CITY
        , d.COUNTERPARTY_RATING
        , d.CREDIT_GRADE
        , d.CUSTOMER_ARM_CODE
        , d.CUSTOMER_NUMBER
        , d.CUSTOMER_OPENED_DATE
        , d.CUSTOMER_SEGMENT_CODE
        , d.CUSTOMER_SUBSEGMENT_CODE
        , d.CUSTOMER_TYPE
        , d.DATE_OF_BIRTH
        , d.DATE_OF_EST
        , d.DECLARED_SALARY
        , d.EMPLOYER_CODE
        , d.INACTIVE
        , d.INDUSTRY_RISK_CODE
        , d.INDUSTRY_SUB_SEGMENT_CODE
        , d.ISLAMIC_BANKING_CUSTOMER
        , d.LANGUAGE_CODE
        , d.MARITAL_STATUS
        , d.NATIONALITY
        , d.PROFESSION
        , d.RESIDENCE_COUNTRY
        , d.RISK_COUNTRY
        , d.RISK_WEIGHTAGE_CODE
        , d.RM_TYPE
        , d.SEX
    FROM mis_automation.silver_fin_db.dim_customer d
    INNER JOIN cif_filter c ON d.CUSTOMER_NUMBER = c.CIF
""").toPandas()

# Re-enable Arrow for other operations
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
demo.shape

In [0]:
# Merge df_merged with demo on CIF (CUSTOMER_NUMBER in demo)
# Keep all demo columns + target from df_merged, deduplicate
df_demo_target = (
    df_merged[['CIF', 'target']]
    .drop_duplicates()
    .merge(demo, left_on='CIF', right_on='CUSTOMER_NUMBER', how='inner')
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Shape: {df_demo_target.shape}")
print(f"Unique customers: {df_demo_target['CIF'].nunique()}")
print(f"\nTarget distribution:")
print(df_demo_target['target'].value_counts(dropna=False))
df_demo_target.head()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# === Step 1: Feature Engineering — Derive age, tenure, business_age ===
reference_date = pd.Timestamp.now()

df_demo_target['CUSTOMER_OPENED_DATE'] = pd.to_datetime(df_demo_target['CUSTOMER_OPENED_DATE'], errors='coerce')
df_demo_target['DATE_OF_BIRTH'] = pd.to_datetime(df_demo_target['DATE_OF_BIRTH'], errors='coerce')
df_demo_target['DATE_OF_EST'] = pd.to_datetime(df_demo_target['DATE_OF_EST'], errors='coerce')

df_demo_target['tenure_years'] = (reference_date - df_demo_target['CUSTOMER_OPENED_DATE']).dt.days / 365.25
df_demo_target['age'] = (reference_date - df_demo_target['DATE_OF_BIRTH']).dt.days / 365.25
df_demo_target['business_age'] = (reference_date - df_demo_target['DATE_OF_EST']).dt.days / 365.25

# Cap unreasonable ages (e.g. > 120 or < 0)
df_demo_target.loc[df_demo_target['age'] > 120, 'age'] = np.nan
df_demo_target.loc[df_demo_target['age'] < 0, 'age'] = np.nan
df_demo_target.loc[df_demo_target['tenure_years'] < 0, 'tenure_years'] = np.nan
df_demo_target.loc[df_demo_target['business_age'] < 0, 'business_age'] = np.nan

# === Step 2: Data Quality Check ===
# Exclude non-feature columns from quality checks
exclude_cols = ['CIF', 'CUSTOMER_NUMBER', 'CUSTOMER_OPENED_DATE', 'DATE_OF_BIRTH', 'DATE_OF_EST', 'target']
candidate_cols = [c for c in df_demo_target.columns if c not in exclude_cols]

# Build a quality summary for each candidate column
quality_records = []
for col in candidate_cols:
    n_total = len(df_demo_target)
    n_missing = df_demo_target[col].isna().sum()
    n_unique = df_demo_target[col].nunique(dropna=True)
    missing_pct = n_missing / n_total * 100
    quality_records.append({
        'column': col,
        'dtype': str(df_demo_target[col].dtype),
        'n_missing': n_missing,
        'missing_pct': round(missing_pct, 2),
        'n_unique': n_unique
    })

df_quality = pd.DataFrame(quality_records).sort_values('missing_pct', ascending=False)
print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)
print(df_quality.to_string(index=False))

# --- Define thresholds ---
MISSING_THRESHOLD = 80   # Drop columns with > 80% missing
MIN_UNIQUE = 2           # Drop columns with only 1 unique value (no variance)

cols_to_drop = df_quality.loc[
    (df_quality['missing_pct'] > MISSING_THRESHOLD) | (df_quality['n_unique'] < MIN_UNIQUE),
    'column'
].tolist()

print(f"\n{'=' * 70}")
print(f"COLUMNS TO DROP (missing > {MISSING_THRESHOLD}% OR unique values < {MIN_UNIQUE}):")
print(f"{'=' * 70}")
if cols_to_drop:
    for c in cols_to_drop:
        row = df_quality.loc[df_quality['column'] == c].iloc[0]
        print(f"  - {c:40s}  missing={row['missing_pct']:.1f}%  unique={row['n_unique']}")
else:
    print("  None — all columns pass quality checks.")

# Drop low-quality columns
df_demo_target.drop(columns=[c for c in cols_to_drop if c in df_demo_target.columns], inplace=True)

# === Step 3: Auto-detect numeric vs categorical features from remaining data ===
remaining_cols = [c for c in df_demo_target.columns if c not in exclude_cols]

numeric_features = []
categorical_features = []

for col in remaining_cols:
    if df_demo_target[col].dtype in ['float64', 'float32', 'int64', 'int32']:
        # Treat as numeric only if it has a reasonable number of distinct values
        if df_demo_target[col].nunique(dropna=True) > 15:
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    else:
        categorical_features.append(col)

all_features = numeric_features + categorical_features
target_col = 'target'

print(f"\n{'=' * 70}")
print(f"FEATURE LISTS (auto-detected from data)")
print(f"{'=' * 70}")
print(f"\nNumeric features ({len(numeric_features)}):")
for f in numeric_features:
    print(f"  - {f}")
print(f"\nCategorical features ({len(categorical_features)}):")
for f in categorical_features:
    print(f"  - {f}")
print(f"\nTotal features: {len(all_features)}")
print(f"Dataset shape after cleanup: {df_demo_target.shape}")



In [0]:
# Ensure target is numeric
df_demo_target[target_col] = pd.to_numeric(df_demo_target[target_col], errors='coerce')
df_analysis = df_demo_target[df_demo_target[target_col].isin([0, 1])].copy()

print(f"Analysis dataset shape: {df_analysis.shape}")
print(f"Target distribution:\n{df_analysis[target_col].value_counts()}")
print(f"\nBad rate: {df_analysis[target_col].mean():.4%}")

# ========================================================================
# 1. NUMERIC FEATURES — Univariate Analysis
# ========================================================================

# --- 1a. Summary statistics ---
print("\n" + "="*70)
print("NUMERIC FEATURES — Univariate Summary Statistics")
print("="*70)
for feat in numeric_features:
    series = df_analysis[feat].dropna().astype(float)
    print(f"\n--- {feat} ---")
    print(f"  count   = {series.count()}")
    print(f"  missing = {df_analysis[feat].isna().sum()} ({df_analysis[feat].isna().mean():.2%})")
    print(f"  mean    = {series.mean():.2f}")
    print(f"  std     = {series.std():.2f}")
    print(f"  min     = {series.min():.2f}")
    print(f"  25%     = {series.quantile(0.25):.2f}")
    print(f"  50%     = {series.median():.2f}")
    print(f"  75%     = {series.quantile(0.75):.2f}")
    print(f"  max     = {series.max():.2f}")
    print(f"  skew    = {series.skew():.2f}")
    print(f"  kurtosis= {series.kurtosis():.2f}")

# --- 1b. Histograms with KDE ---
n_num = len(numeric_features)
ncols = min(n_num, 2)
nrows = (n_num + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5 * nrows))
if n_num == 1:
    axes = np.array([axes])
axes = axes.flatten()
for i, feat in enumerate(numeric_features):
    ax = axes[i]
    series = df_analysis[feat].dropna().astype(float)
    ax.hist(series, bins=50, color='steelblue', edgecolor='white', alpha=0.7, density=True)
    try:
        series.plot.kde(ax=ax, color='crimson', linewidth=2)
    except Exception:
        pass
    ax.axvline(series.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Mean={series.mean():.1f}')
    ax.axvline(series.median(), color='green', linestyle='--', linewidth=1.5, label=f'Median={series.median():.1f}')
    ax.set_title(f'{feat} — Distribution', fontsize=13)
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numeric Features — Univariate Distributions', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# --- 1c. Box plots ---
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5 * nrows))
if n_num == 1:
    axes = np.array([axes])
axes = axes.flatten()
for i, feat in enumerate(numeric_features):
    ax = axes[i]
    series = df_analysis[feat].dropna().astype(float)
    bp = ax.boxplot(series, vert=True, patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.6)
    ax.set_title(f'{feat} — Box Plot', fontsize=13)
    ax.set_ylabel(feat)
    ax.annotate(f'med={series.median():.1f}', xy=(1, series.median()),
                fontsize=10, ha='center', va='bottom', color='red')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numeric Features — Box Plots (Outlier Detection)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# ========================================================================
# 2. CATEGORICAL FEATURES — Univariate Analysis
# ========================================================================
print("\n" + "="*70)
print("CATEGORICAL FEATURES — Univariate Frequency Tables")
print("="*70)

MAX_CATEGORIES_PLOT = 20

for feat in categorical_features:
    df_analysis[feat] = df_analysis[feat].astype(str)
    freq = df_analysis[feat].value_counts()
    freq_pct = df_analysis[feat].value_counts(normalize=True) * 100
    n_unique = freq.shape[0]
    n_missing = (df_analysis[feat].isin(['nan', 'None', ''])).sum()

    print(f"\n--- {feat} (unique: {n_unique}, missing/blank: {n_missing}) ---")
    freq_table = pd.DataFrame({'count': freq, 'pct': freq_pct.round(2)})
    print(freq_table.head(MAX_CATEGORIES_PLOT).to_string())

    # Bar chart for top categories
    plot_data = freq.head(MAX_CATEGORIES_PLOT)
    if len(plot_data) > 0:
        fig, ax = plt.subplots(figsize=(max(10, len(plot_data) * 0.6), 5))
        bars = ax.bar(range(len(plot_data)), plot_data.values, color='steelblue', alpha=0.7, edgecolor='white')
        ax.set_xticks(range(len(plot_data)))
        ax.set_xticklabels(plot_data.index, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Count')
        ax.set_xlabel(feat)
        ax.set_title(f'{feat} — Frequency Distribution'
                     f'{" (Top " + str(MAX_CATEGORIES_PLOT) + ")" if n_unique > MAX_CATEGORIES_PLOT else ""}',
                     fontsize=13)
        # Annotate percentages on bars
        total = plot_data.values.sum()
        for idx, (bar, val) in enumerate(zip(bars, plot_data.values)):
            pct = val / df_analysis.shape[0] * 100
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
        plt.tight_layout()
        plt.show()

print("\n" + "="*70)
print("Univariate analysis complete.")
print("="*70)

In [0]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt

SEED = 42

# ── 1. Prepare features and target ──────────────────────────────────────────
feature_cols = [
    'tenure_years', 'age',
    'CUSTOMER_SUBSEGMENT_CODE', 'DECLARED_SALARY',
    'EMPLOYER_CODE', 'ISLAMIC_BANKING_CUSTOMER',
    'MARITAL_STATUS', 'SEX'
]

numeric_features = ['tenure_years', 'age', 'DECLARED_SALARY']
categorical_features = ['CUSTOMER_SUBSEGMENT_CODE', 'EMPLOYER_CODE',
                        'ISLAMIC_BANKING_CUSTOMER', 'MARITAL_STATUS', 'SEX']

df_model = df_demo_target[feature_cols + ['target']].copy()
df_model['target'] = pd.to_numeric(df_model['target'], errors='coerce')
df_model = df_model[df_model['target'].isin([0, 1])].reset_index(drop=True)

X = df_model[feature_cols]
y = df_model['target'].astype(int)

# ── 2. Check class imbalance ────────────────────────────────────────────────
minority_share = y.value_counts(normalize=True).min()
print(f"Class distribution:\n{y.value_counts()}")
print(f"Minority class share: {minority_share:.2%}")

# ── 3. Train / test split (stratified) ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"\nTrain shape: {X_train.shape}  |  Test shape: {X_test.shape}")
print(f"Train bad rate: {y_train.mean():.4%}  |  Test bad rate: {y_test.mean():.4%}")

# ── 4. Preprocessing pipeline ───────────────────────────────────────────────
# Cast categorical columns to string so OrdinalEncoder handles them uniformly
for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ── 5. Determine class weighting ────────────────────────────────────────────
is_imbalanced = minority_share < 0.20
if is_imbalanced:
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    spw = n_neg / n_pos
    print(f"\nClass imbalance detected — minority class is ~{minority_share:.1%} of rows.")
    print(f"Using scale_pos_weight = {spw:.2f}")
else:
    spw = 1.0

# ── 6. Build full pipeline (preprocessor + LightGBM) ───────────────────────
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lgbm_clf)
])

# ── 7. Train ────────────────────────────────────────────────────────────────
pipeline.fit(X_train, y_train)

# ── 8. Evaluate on test set ─────────────────────────────────────────────────
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"\n{'='*60}")
print("TEST SET EVALUATION")
print(f"{'='*60}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# ── 9. Plots ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="LightGBM")
axes[0].set_title("ROC Curve")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="LightGBM")
axes[1].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

# Feature importance
booster = pipeline.named_steps['classifier']
feat_names = numeric_features + categorical_features
importances = booster.feature_importances_
fi = pd.DataFrame({'feature': feat_names, 'importance': importances}) \
       .sort_values('importance', ascending=True)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.barh(fi['feature'], fi['importance'], color='steelblue')
ax2.set_title('LightGBM Feature Importance')
ax2.set_xlabel('Importance (split count)')
plt.tight_layout()
plt.show()

In [0]:
import shap
import matplotlib.pyplot as plt

# Compute SHAP values using the trained pipeline
# Transform training data through the preprocessor
X_train_transformed = pipeline.named_steps['preprocessor'].transform(X_train)

# Create a SHAP TreeExplainer on the LightGBM classifier
explainer = shap.TreeExplainer(pipeline.named_steps['classifier'])
shap_values = explainer(X_train_transformed)

# Map feature names after preprocessing
feat_names = numeric_features + categorical_features
shap_values.feature_names = feat_names

# Get target values aligned with X_train
target_values = y_train.values

# SHAP scatter plots for tenure_years, age, DECLARED_SALARY colored by target
features_to_plot = ['tenure_years', 'age', 'DECLARED_SALARY']

for feat in features_to_plot:
    feat_idx = feat_names.index(feat)
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.plots.scatter(
        shap_values[:, feat_idx],
        color=target_values,
        ax=ax,
        show=False
    )
    ax.set_title(f'SHAP Scatter — {feat} (colored by target)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [0]:
# df_pd.CREDITCARDNO.nunique()#188667632
df_pd.REFERENCE_NUMBER.nunique()

In [0]:
def load_aecb_feature_store(df_pd):
    """
    Load AECB feature store data filtered to the reference numbers present
    in the cleaned credit card dataset.

    Steps:
      1. Query the full AECB credit card feature store table.
      2. Extract unique reference numbers from the cleaned credit card DataFrame.
      3. Filter the AECB data to only include matching reference numbers.
      4. Convert the filtered Spark DataFrame to a Pandas DataFrame.

    Args:
        df_pd (pd.DataFrame): Cleaned credit card Pandas DataFrame containing
                              a 'REFERENCE_NUMBER' column.

    Returns:
        pd.DataFrame: AECB feature store data filtered to relevant reference numbers.
    """

    # --- Step 1: Load full AECB feature store table ----------------------------
    aecb = spark.sql("""
        SELECT *
        FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
    """)

    # --- Step 2: Get unique reference numbers from cleaned data ----------------
    ref_numbers = df_pd.REFERENCE_NUMBER.unique().tolist()

    # --- Step 2b: Ensure reference numbers are strings for consistent matching -
    ref_numbers = [str(r) for r in ref_numbers]

    # --- Step 3: Filter AECB data to matching reference numbers ----------------
    aecb_filtered = aecb.withColumn("Reference_Number", F.col("Reference_Number").cast("string")) \
                        .filter(F.col("Reference_Number").isin(ref_numbers))

    # --- Step 4: Convert to Pandas and return ----------------------------------
    aecb_pd = aecb_filtered.toPandas()

    return aecb_pd


# Execute and inspect
aecb_pd = load_aecb_feature_store(df_ref)
print("AECB shape:", aecb_pd.shape)

# Ensure both join keys are the same dtype (string) before merging
aecb_pd['Reference_Number'] = aecb_pd['Reference_Number'].astype(str)
df_ref['REFERENCE_NUMBER'] = df_ref['REFERENCE_NUMBER'].astype(str)

# Merge AECB data with df_ref to bring in the target column
aecb_merged = aecb_pd.merge(
    df_ref[['REFERENCE_NUMBER', 'target']].drop_duplicates(),
    left_on='Reference_Number',
    right_on='REFERENCE_NUMBER',
    how='inner'
)

print("Merged shape:", aecb_merged.shape)
print(f"\nTarget distribution:\n{aecb_merged['target'].value_counts(dropna=False)}")
aecb_merged.head()

In [0]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
target_col = 'target'

# ══════════════════════════════════════════════════════════════════════════════
# 1. INITIAL INSPECTION
# ══════════════════════════════════════════════════════════════════════════════
print(f"aecb_merged shape: {aecb_merged.shape}")
print(f"\nColumn dtypes:\n{aecb_merged.dtypes.to_string()}")
print(f"\nTarget distribution:\n{aecb_merged[target_col].value_counts(dropna=False)}")
print(f"Bad rate: {aecb_merged[target_col].mean():.4%}")

# ══════════════════════════════════════════════════════════════════════════════
# 2. EXCLUDE NON-FEATURE / ID / KEY COLUMNS
# ══════════════════════════════════════════════════════════════════════════════
# Columns that are identifiers, keys, dates, or free-text — never useful as features
exclude_patterns = [
    'CIF', 'CUSTOMER_NUMBER', 'Reference_Number', 'REFERENCE_NUMBER',
    'CREDITCARDNO', 'CRNNO', 'ACCOUNT', 'CONTRACT',
    'target'   # target itself
]
# Also drop any pure-date columns (they need engineering, not raw feeding)
date_cols = aecb_merged.select_dtypes(include=['datetime64', 'datetimetz']).columns.tolist()

cols_to_exclude = [c for c in aecb_merged.columns
                   if any(pat.lower() in c.lower() for pat in exclude_patterns)]
cols_to_exclude = list(set(cols_to_exclude + date_cols))

candidate_cols = [c for c in aecb_merged.columns if c not in cols_to_exclude]
print(f"\nExcluded columns ({len(cols_to_exclude)}): {cols_to_exclude}")
print(f"Candidate feature columns: {len(candidate_cols)}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. DATA QUALITY ASSESSMENT
# ══════════════════════════════════════════════════════════════════════════════
MISSING_THRESHOLD = 80   # drop if > 80 % missing
MIN_UNIQUE = 2           # drop if only 1 unique value (zero variance)
CARDINALITY_CAP = 500    # cap for categorical columns (too many levels → noise)

quality_records = []
for col in candidate_cols:
    n_total = len(aecb_merged)
    n_missing = aecb_merged[col].isna().sum()
    n_unique = aecb_merged[col].nunique(dropna=True)
    missing_pct = n_missing / n_total * 100
    quality_records.append({
        'column': col,
        'dtype': str(aecb_merged[col].dtype),
        'n_missing': n_missing,
        'missing_pct': round(missing_pct, 2),
        'n_unique': n_unique
    })

df_quality = pd.DataFrame(quality_records).sort_values('missing_pct', ascending=False)
print("\n" + "=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)
print(df_quality.to_string(index=False))

# --- Identify columns to drop ---
cols_high_missing = df_quality.loc[df_quality['missing_pct'] > MISSING_THRESHOLD, 'column'].tolist()
cols_no_variance  = df_quality.loc[df_quality['n_unique'] < MIN_UNIQUE, 'column'].tolist()

# Object columns with cardinality > cap (too many unique values — likely free text or IDs)
cols_high_card = df_quality.loc[
    (df_quality['dtype'] == 'object') & (df_quality['n_unique'] > CARDINALITY_CAP),
    'column'
].tolist()

cols_to_drop = list(set(cols_high_missing + cols_no_variance + cols_high_card))

print(f"\n{'=' * 80}")
print(f"COLUMNS TO DROP")
print(f"{'=' * 80}")
for c in cols_to_drop:
    row = df_quality.loc[df_quality['column'] == c].iloc[0]
    reason = []
    if c in cols_high_missing:
        reason.append(f"missing={row['missing_pct']:.1f}%")
    if c in cols_no_variance:
        reason.append(f"unique={row['n_unique']}")
    if c in cols_high_card:
        reason.append(f"high_cardinality={row['n_unique']}")
    print(f"  ✗ {c:45s}  {', '.join(reason)}")

# Surviving features
feature_cols = [c for c in candidate_cols if c not in cols_to_drop]
print(f"\nSurviving features: {len(feature_cols)}")

# ══════════════════════════════════════════════════════════════════════════════
# 4. AUTO-DETECT NUMERIC vs CATEGORICAL + DTYPE COERCION
# ══════════════════════════════════════════════════════════════════════════════
numeric_features = []
categorical_features = []

for col in feature_cols:
    # Try to coerce to numeric; if most values convert, treat as numeric
    if aecb_merged[col].dtype in ['float64', 'float32', 'int64', 'int32']:
        if aecb_merged[col].nunique(dropna=True) > 15:
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    elif aecb_merged[col].dtype == 'object':
        # Attempt numeric coercion — some numeric cols are stored as strings
        coerced = pd.to_numeric(aecb_merged[col], errors='coerce')
        non_null_original = aecb_merged[col].notna().sum()
        non_null_coerced  = coerced.notna().sum()
        if non_null_original > 0 and (non_null_coerced / non_null_original) > 0.90:
            # Predominantly numeric — coerce in place
            aecb_merged[col] = coerced
            if coerced.nunique(dropna=True) > 15:
                numeric_features.append(col)
            else:
                categorical_features.append(col)
        else:
            categorical_features.append(col)
    else:
        categorical_features.append(col)

all_features = numeric_features + categorical_features

print(f"\n{'=' * 80}")
print(f"FEATURE LISTS (auto-detected + dtype-coerced)")
print(f"{'=' * 80}")
print(f"\nNumeric features ({len(numeric_features)}):")
for f in numeric_features:
    print(f"  • {f}  (dtype={aecb_merged[f].dtype}, unique={aecb_merged[f].nunique()})")
print(f"\nCategorical features ({len(categorical_features)}):")
for f in categorical_features:
    print(f"  • {f}  (dtype={aecb_merged[f].dtype}, unique={aecb_merged[f].nunique()})")
print(f"\nTotal features: {len(all_features)}")

# ══════════════════════════════════════════════════════════════════════════════
# 5. TARGET LEAKAGE QUICK SCREEN
# ══════════════════════════════════════════════════════════════════════════════
aecb_merged[target_col] = pd.to_numeric(aecb_merged[target_col], errors='coerce')
df_model = aecb_merged[aecb_merged[target_col].isin([0, 1])].copy()

if numeric_features:
    corr_with_target = df_model[numeric_features + [target_col]].corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
    print(f"\n{'=' * 80}")
    print("LEAKAGE SCREEN — |Correlation| with target (numeric features)")
    print(f"{'=' * 80}")
    print(corr_with_target.to_string())
    suspect = corr_with_target[corr_with_target > 0.95]
    if len(suspect) > 0:
        print(f"\n⚠️  HIGH-CORRELATION SUSPECTS (|r| > 0.95) — dropping to avoid leakage:")
        for s in suspect.index:
            print(f"  ✗ {s}  (|r| = {suspect[s]:.4f})")
        all_features = [f for f in all_features if f not in suspect.index]
        numeric_features = [f for f in numeric_features if f not in suspect.index]
    else:
        print("\n✓ No features with suspiciously high correlation detected.")

# ══════════════════════════════════════════════════════════════════════════════
# 6. CORRELATION HEATMAP (numeric features)
# ══════════════════════════════════════════════════════════════════════════════
if len(numeric_features) >= 2:
    fig, ax = plt.subplots(figsize=(min(20, len(numeric_features)), min(16, len(numeric_features) * 0.8)))
    corr_matrix = df_model[numeric_features].corr()
    sns.heatmap(corr_matrix, annot=len(numeric_features) <= 20, fmt='.2f',
                cmap='coolwarm', center=0, ax=ax, linewidths=0.5)
    ax.set_title('Numeric Feature Correlation Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. PREPARE DATA & BUILD LIGHTGBM PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
final_features = numeric_features + categorical_features
print(f"\n{'=' * 80}")
print(f"FINAL FEATURE SET FOR MODEL ({len(final_features)} features)")
print(f"{'=' * 80}")
for f in final_features:
    print(f"  • {f}")

X = df_model[final_features].copy()
y = df_model[target_col].astype(int)

# Cast categorical columns to string
for col in categorical_features:
    X[col] = X[col].astype(str)

# ── Class imbalance ──
minority_share = y.value_counts(normalize=True).min()
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"Minority class share: {minority_share:.2%}")

# ── Train / test split (stratified) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain shape: {X_train.shape}  |  Test shape: {X_test.shape}")
print(f"Train bad rate: {y_train.mean():.4%}  |  Test bad rate: {y_test.mean():.4%}")

# ── Preprocessing pipeline ──
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ── Class weighting ──
is_imbalanced = minority_share < 0.20
if is_imbalanced:
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    spw = n_neg / n_pos
    print(f"\nClass imbalance detected — using scale_pos_weight = {spw:.2f}")
else:
    spw = 1.0

# ── LightGBM pipeline ──
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

pipeline_aecb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lgbm_clf)
])

# ── Train ──
pipeline_aecb.fit(X_train, y_train)

# ══════════════════════════════════════════════════════════════════════════════
# 8. EVALUATE ON TEST SET
# ══════════════════════════════════════════════════════════════════════════════
y_pred = pipeline_aecb.predict(X_test)
y_prob = pipeline_aecb.predict_proba(X_test)[:, 1]

acc    = accuracy_score(y_test, y_pred)
prec   = precision_score(y_test, y_pred, zero_division=0)
rec    = recall_score(y_test, y_pred, zero_division=0)
f1     = f1_score(y_test, y_pred, zero_division=0)
auc    = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"\n{'=' * 60}")
print("AECB MODEL — TEST SET EVALUATION")
print(f"{'=' * 60}")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, zero_division=0)}")

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="LightGBM (AECB)")
axes[0].set_title("ROC Curve")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="LightGBM (AECB)")
axes[1].set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

# ── Feature importance ──
booster = pipeline_aecb.named_steps['classifier']
importances = booster.feature_importances_
fi = pd.DataFrame({'feature': final_features, 'importance': importances}) \
       .sort_values('importance', ascending=True)

fig2, ax2 = plt.subplots(figsize=(10, max(5, len(final_features) * 0.35)))
ax2.barh(fi['feature'], fi['importance'], color='steelblue')
ax2.set_title('LightGBM Feature Importance (AECB Features)', fontsize=13)
ax2.set_xlabel('Importance (split count)')
plt.tight_layout()
plt.show()

In [0]:
!pip install shap
import shap
import pandas as pd
import matplotlib.pyplot as plt

# Transform training data through the preprocessor
X_train_transformed = pipeline_aecb.named_steps['preprocessor'].transform(X_train)

# Create SHAP TreeExplainer on the LightGBM classifier
explainer = shap.TreeExplainer(pipeline_aecb.named_steps['classifier'])
shap_values = explainer(X_train_transformed)

# Map feature names after preprocessing
feat_names = numeric_features + categorical_features
shap_values.feature_names = feat_names

# Get target values aligned with X_train
target_values = y_train.values

# SHAP scatter plots for aecb_score and mob_first_cntrct colored by target
features_to_plot = ['aecb_score', 'mob_first_cntrct']

for feat in features_to_plot:
    feat_idx = feat_names.index(feat)
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.plots.scatter(
        shap_values[:, feat_idx],
        color=target_values,
        ax=ax,
        show=False
    )
    ax.set_title(f'SHAP Scatter — {feat} (colored by target)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [0]:
AV = AutoViz_Class()
dft = AV.AutoViz(
    filename="",
    dfte=aecb_pd,
    depVar="",
    chart_format="png",
    max_rows_analyzed=100000,
    max_cols_analyzed=30
)

In [0]:
from pyspark.sql.functions import col
# df.filter(~col("APPROVEDDT").eqNullSafe(col("APPROVEDATE"))).count()

df.groupBy("EMPLOYMENT_CATEGORY").count().orderBy("count", ascending=False).show()

In [0]:
# --- Insight: One REFERENCE_NUMBER can be associated with multiple CREDITCARDNOs ---
# A single customer (identified by CRNNO) may apply for multiple credit cards,
# sometimes even on the same day. This means modelling per CREDITCARDNO would treat
# each card independently, ignoring the shared customer context.
#
# Decision:
#   - We do NOT build the model per CREDITCARDNO.
#   - Instead, we use REFERENCE_NUMBER as the modelling grain, which represents
#     a unique combination of the credit-card APPLICATION and the customer.
#   - When a customer applies for several cards within the same month, those
#     applications share the same AECB (Al Etihad Credit Bureau) report.
#     Therefore, we build ONE model per AECB record / REFERENCE_NUMBER,
#     leveraging the fact that the bureau snapshot is identical across those
#     concurrent applications.
#
# The query below surfaces REFERENCE_NUMBERs linked to more than one card,
# confirming this many-to-one relationship in the data.
# ---------------------------------------------------------------------------
shared_ref = df_pd.groupby('REFERENCE_NUMBER')['CREDITCARDNO'].nunique().loc[lambda x: x > 1]
df_pd[df_pd['REFERENCE_NUMBER'].isin(shared_ref.index)]

df_pd.loc[df_pd['REFERENCE_NUMBER']=='1589992'][['SNAPSHOT_DATE','CRNNO','CREDITCARDNO','EMPLOYMENT_CATEGORY','PRIMARYCARD','APPROVEDATE','CREDITLIMIT','LASTSTATEBALTOTAL','REFERENCE_NUMBER']]

In [0]:
# "5239267539247004" in creditcards_less_than_3_open
df_pd.loc[df_pd['CREDITCARDNO']=="5171466077654965"]
data = df_pd[cols]                    


In [0]:
tmp = df.filter(F.col("CREDITCARDNO") == "5171466260553586").toPandas()
tmp[["SNAPSHOT_DATE","CREDITCARDNO","CRNNO","EMPLOYMENT_CATEGORY","APPROVEDATE","LOS","open_count","CURRENT_STATUS"]].sort_values("SNAPSHOT_DATE")

In [0]:
data.describe()

In [0]:
data[cols].groupby(['CREDITCARDNO','APPROVEDATE']).agg(los=('SNAPSHOT_DATE',pd.Series.nunique)).reset_index()

In [0]:
df[cols].loc[df.CREDITCARDNO=="1531000000370110"].sort_values("SNAPSHOT_DATE")#"1531000004671703"

In [0]:
df_del = df_pd[["CREDITCARDNO","SNAPSHOT_DATE",'DELINQUENCYBUCKET','APPROVEDATE']].groupby(["CREDITCARDNO"]).agg(max_delinquencybucket=('DELINQUENCYBUCKET',max),min_approvedate=('APPROVEDATE',min),max_snapshotdate=('SNAPSHOT_DATE',max)).reset_index()
df_del["target_12m_date"] = df_del['min_approvedate']+pd.DateOffset(months=12)

In [0]:
df_del.head()


In [0]:
df_del.loc[df_del.target_12m_date < df_del.max_snapshotdate]

In [0]:
import matplotlib.pyplot as plt
df_del['max_delinquencybucket'] = pd.to_numeric(df_del['max_delinquencybucket'], errors='coerce').fillna(-1).astype(int)
df_del['max_delinquencybucket'].value_counts().sort_index()
df_del['max_delinquencybucket'].loc[df_del['max_delinquencybucket']>0].value_counts().sort_index().plot(kind="bar",figsize=(8,4))
plt.title("Distibution of Deliquency Buckets")
plt.xlabel("Bucket")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [0]:
!pip install --upgrade optuna statsmodels scikit-learn gputil dvc apache-airflow lightgbm xgboost catboost h2o python-tds



In [0]:
dff = prepare_data(data, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                  bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE")
dff.head()

In [0]:
transition_rate, transition_counts = build_roll_rate_matrix(dff, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE", bucket_col="DELINQUENCYBUCKET", n_periods=1)
transition_rate.head()

In [0]:
transition_rate, transition_counts = build_roll_rate_matrix(dff, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE", bucket_col="DELINQUENCYBUCKET", n_periods=1)
transition_rate.head()

In [0]:
n_periods = 1

# Professional style
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100, aspect="auto")

# Axis ticks and labels
ax.set_xticks(range(len(transition_rate.columns)))
ax.set_yticks(range(len(transition_rate.index)))
ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns], fontsize=10, fontweight='medium', rotation=45, ha='right')
ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index], fontsize=10, fontweight='medium')

# Move x-tick labels to bottom and ensure visibility
ax.tick_params(axis='x', bottom=True, top=False, labelbottom=True, labeltop=False, pad=8)
ax.tick_params(axis='y', pad=5)

# Axis labels
ax.set_xlabel(f"Bucket after {n_periods} month(s)", fontsize=12, fontweight='bold', labelpad=12)
ax.set_ylabel("Current Bucket", fontsize=12, fontweight='bold', labelpad=10)

# Title
ax.set_title(f"Roll Rate Matrix ({n_periods}-Month Transition, %)", fontsize=14, fontweight='bold', pad=15)

# Annotate cells with values
for i in range(transition_rate.shape[0]):
    for j in range(transition_rate.shape[1]):
        
        value = transition_rate.values[i, j]
        # Use white text on dark cells for readability
        text_color = "white" if value > 60 else "black"
        ax.text(j, i, f"{value:.1f}",
                ha="center", va="center", fontsize=10, fontweight='semibold', color=text_color)

# Colorbar
cbar = fig.colorbar(im, ax=ax, label="Transition Probability (%)", shrink=0.85)
cbar.ax.tick_params(labelsize=9)
cbar.set_label("Transition Probability (%)", fontsize=11)

# Remove gridlines from imshow
ax.grid(False)

plt.tight_layout()
plt.show()

In [0]:
transition_counts

In [0]:
transition_counts

In [0]:
transition_counts

In [0]:
# rate, total = eventual_roll_forward_rate(dff, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
#                                 bucket_col="DELINQUENCYBUCKET",
#                                 from_bucket=2, to_bucket_min=4, within_months=6)
# rate.head(4)
# total.head(4)
account_summary, bad_definitions = define_bad_flags(dff,id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET", mob_col="MOB", bad_definitions=None)
account_summary.head()
# 
                        

In [0]:
summary_df, overlap_df = compare_bad_definitions(account_summary, bad_definitions)
overlap_df

In [0]:
account_summary.head()

In [0]:
# Filter out vintages greater than 2024-07 to keep only mature cohorts
dff_mature = dff[dff["_approve_month"] < pd.Timestamp("2024-07-01")]
account_summary_mature = account_summary[account_summary["approve_month"] < pd.Timestamp("2024-07-01")]

vintage_cum_bad_rate = vintage_cumulative_bad_rate(dff_mature, account_summary_mature, id_col="CREDITCARDNO",
                                 bucket_col="DELINQUENCYBUCKET", min_bucket=3,
                                 mob_col="MOB", max_mob=24)
vintage_cum_bad_rate_ = vintage_cum_bad_rate.dropna()

In [0]:
vintage_cum_bad_rate_.loc['2022-08']

In [0]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 7))

cmap = plt.cm.viridis
colors = [cmap(i / max(len(vintage_cum_bad_rate_.index) - 1, 1)) for i in range(len(vintage_cum_bad_rate_.index))]

for idx, vintage in enumerate(vintage_cum_bad_rate_.index):
    series = vintage_cum_bad_rate_.loc[vintage].dropna()
    ax.plot(series.index, series.values, marker="o", markersize=4, linewidth=1.8,
            color=colors[idx], label=vintage)

ax.set_xlabel("Months on Book (MOB)", fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel("Cumulative Bad Rate (%)", fontsize=12, fontweight='bold', labelpad=10)
ax.set_title("Vintage Cumulative Bad Rate Analysis", fontsize=14, fontweight='bold', pad=15)
ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8,
          title_fontsize=10, frameon=True, framealpha=0.9)
ax.grid(alpha=0.3, linestyle='--')
ax.tick_params(axis='both', labelsize=10)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [0]:
mature_mob, avg_curve, marginal = find_maturity_point(vintage_cum_bad_rate, marginal_threshold=0.3)
mature_mob, avg_curve, marginal

In [0]:
"""
Roll Rate & Vintage Analysis — 用于确定信用卡 Bad 定义 (Target Variable)
=========================================================================

场景：你有逐月快照数据，字段包括
    CREDITCARDNO      : 卡号（如果你的原始表字段是 CREDTICARDNO，直接改 id_col 参数即可）
    SNAPSHOT_DATE      : 月度快照日期
    DELINQUENCYBUCKET  : 0=正常, 1=1-29天逾期, 2=30-59天逾期, 3=60-89天逾期, 4=90+天逾期
                         （按你实际业务口径调整 bucket 数值含义）
    APPROVEDATE     : 开卡/激活日期

目标：通过
    1) Roll Rate 分析 —— 一旦进入某个逾期阶段，未来滚到更差阶段的概率有多大
    2) Vintage 分析   —— 不同开卡批次(Vintage)的累计坏账率随账龄(MOB)如何演化，
                         曲线什么时候开始走平（即"成熟点"）
来帮你在多个候选 Bad 定义（如 Ever60DPD_12M / Ever30DPD_6M / Ever90DPD_12M）之间做选择。
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 0. 数据准备：计算 MOB (Months on Book)
# =============================================================================

def prepare_data(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                  bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE"):
    """
    清洗 + 计算 MOB。返回处理后的 long-format dataframe，
    新增列: _snap_month, _approve_month, MOB
    """
    df = df.copy()
    df[snap_col] = pd.to_datetime(df[snap_col])
    df[approve_col] = pd.to_datetime(df[approve_col])

    # 统一取月初，避免同月内日期不一致导致 MOB 计算错位
    df["_snap_month"] = df[snap_col].values.astype("datetime64[M]")
    df["_approve_month"] = df[approve_col].values.astype("datetime64[M]")

    df["MOB"] = (
        (df["_snap_month"].dt.year - df["_approve_month"].dt.year) * 12
        + (df["_snap_month"].dt.month - df["_approve_month"].dt.month)
    )

    before = len(df)
    df = df[df["MOB"] >= 0].copy()  # 剔除快照早于开户日的脏数据
    dropped = before - len(df)
    if dropped > 0:
        print(f"[prepare_data] 剔除了 {dropped} 条 MOB<0 的异常记录（快照早于开户日）")

    df = df.sort_values([id_col, snap_col]).reset_index(drop=True)
    return df


# =============================================================================
# 1. Roll Rate 分析
# =============================================================================

def build_roll_rate_matrix(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                            bucket_col="DELINQUENCYBUCKET", n_periods=1):
    """
    n_periods 个月的滚动迁移矩阵：
    Row = 当前 Bucket, Column = n_periods 个月后的 Bucket, Value = 迁移概率(%)
    只统计真正间隔恰好 n_periods 个月、且账户在两个时点都有记录的数据对，
    避免因为账户数据缺月造成的虚假跳跃。
    """
    df = df.sort_values([id_col, snap_col]).copy()
    df["_period"] = df[snap_col].dt.to_period("M")

    df["_next_bucket"] = df.groupby(id_col)[bucket_col].shift(-1)
    df["_next_period"] = df.groupby(id_col)["_period"].shift(-1)

    valid = df.dropna(subset=["_next_bucket", "_next_period"]).copy()
    gap = (valid["_next_period"] - valid["_period"]).apply(lambda x: x.n)
    valid = valid[gap == n_periods]

    transition_counts = pd.crosstab(valid[bucket_col], valid["_next_bucket"])
    transition_rate = transition_counts.div(transition_counts.sum(axis=1), axis=0) * 100
    return transition_rate.round(2), transition_counts


def plot_roll_rate_heatmap(transition_rate, n_periods=1, save_path=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(transition_rate.values, cmap="Reds", vmin=0, vmax=100)
    ax.set_xticks(range(len(transition_rate.columns)))
    ax.set_yticks(range(len(transition_rate.index)))
    ax.set_xticklabels([f"Bucket {c}" for c in transition_rate.columns])
    ax.set_yticklabels([f"Bucket {r}" for r in transition_rate.index])
    ax.set_xlabel(f"Bucket after {n_periods} month(s)")
    ax.set_ylabel("Current Bucket")
    ax.set_title(f"Roll Rate Matrix ({n_periods}-Month Transition, %)")
    for i in range(transition_rate.shape[0]):
        for j in range(transition_rate.shape[1]):
            ax.text(j, i, f"{transition_rate.values[i, j]:.1f}",
                    ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="Transition Probability (%)")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def eventual_roll_forward_rate(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                                bucket_col="DELINQUENCYBUCKET",
                                from_bucket=2, to_bucket_min=4, within_months=6):
    """
    核心指标（用来判断某个 DPD 阶段是不是"回不去了"）：
    对所有曾处于 from_bucket 状态的账户-月份观察点，
    计算未来 within_months 个月内"最终"是否曾经达到 >= to_bucket_min 的比例。

    例：from_bucket=2 (30-59dpd), to_bucket_min=4 (90+dpd), within_months=6
    -> "一旦进入 30+ 逾期，未来 6 个月内滚到 90+ 逾期的概率有多大？"
    如果这个比例很高（比如 >80%），说明 30dpd 已经是一个很强的 bad 信号，
    可以考虑把 bad 定义提前到 30dpd，而不必等到 90dpd。

    注意：这里用逐账户循环实现，逻辑清晰但在千万级数据量下会慢，
    大数据量场景建议改用 groupby + rolling window 的向量化写法。
    """
    df = df.sort_values([id_col, snap_col]).copy()
    hits, total = 0, 0
    for _, g in df.groupby(id_col):
        g = g.reset_index(drop=True)
        at_from = g[g[bucket_col] == from_bucket]
        for _, row in at_from.iterrows():
            future = g[(g[snap_col] > row[snap_col]) &
                       (g[snap_col] <= row[snap_col] + pd.DateOffset(months=within_months))]
            if len(future) == 0:
                continue
            total += 1
            if (future[bucket_col] >= to_bucket_min).any():
                hits += 1
    rate = (hits / total * 100) if total > 0 else np.nan
    return rate, total


# =============================================================================
# 2. Bad 定义标注（account-level）
# =============================================================================

def define_bad_flags(df, id_col="CREDITCARDNO", bucket_col="DELINQUENCYBUCKET",
                      mob_col="MOB", bad_definitions=None):
    """
    bad_definitions 示例:
        {
            "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
            "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
            "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
        }
    (min_bucket 按你实际的 DELINQUENCYBUCKET 编码调整；本例假设
     0=正常 1=1-29dpd 2=30-59dpd 3=60-89dpd 4=90+dpd)

    返回 account-level dataframe：每个账户一行，
    标注是否 mature（观察窗口是否已跑满）、是否触发 bad。
    """
    if bad_definitions is None:
        bad_definitions = {
            "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
            "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
            "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
        }

    account_summary = df.groupby(id_col).agg(
        max_mob_observed=("MOB", "max"),
        approve_month=("_approve_month", "first"),
    ).reset_index()

    for name, cfg in bad_definitions.items():
        window = cfg["window_months"]
        min_bucket = cfg["min_bucket"]

        account_summary[f"{name}_mature"] = account_summary["max_mob_observed"] >= window

        window_df = df[df[mob_col] <= window]
        bad_flags = (
            window_df.groupby(id_col)[bucket_col]
            .apply(lambda s: bool((s >= min_bucket).any()))
            .rename(f"{name}_bad")
        )
        account_summary = account_summary.merge(bad_flags, on=id_col, how="left")
        account_summary[f"{name}_bad"] = account_summary[f"{name}_bad"].fillna(False)

    return account_summary, bad_definitions


def compare_bad_definitions(account_summary, bad_definitions):
    """
    对比不同 Bad 定义：
      - 各定义在"成熟样本"口径下的 Bad Rate
      - 两两定义之间的一致性(Agreement%) 和相关性(correlation)
    """
    names = list(bad_definitions.keys())
    rows = []
    for name in names:
        mature = account_summary[f"{name}_mature"]
        pop = account_summary[mature]
        bad_rate = pop[f"{name}_bad"].mean() * 100 if len(pop) else np.nan
        rows.append({
            "definition": name,
            "mature_sample_size": int(mature.sum()),
            "bad_rate_%": round(bad_rate, 2),
        })
    summary_df = pd.DataFrame(rows)

    overlap_rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            n1, n2 = names[i], names[j]
            both_mature = account_summary[f"{n1}_mature"] & account_summary[f"{n2}_mature"]
            sub = account_summary[both_mature]
            if len(sub) == 0:
                continue
            agree = (sub[f"{n1}_bad"] == sub[f"{n2}_bad"]).mean() * 100
            corr = sub[f"{n1}_bad"].astype(int).corr(sub[f"{n2}_bad"].astype(int))
            overlap_rows.append({
                "pair": f"{n1} vs {n2}",
                "sample_size": len(sub),
                "agreement_%": round(agree, 2),
                "correlation": round(corr, 3),
            })
    overlap_df = pd.DataFrame(overlap_rows)
    return summary_df, overlap_df


# =============================================================================
# 3. Vintage 分析
# =============================================================================

def vintage_cumulative_bad_rate(df, account_summary, id_col="CREDITCARDNO",
                                 bucket_col="DELINQUENCYBUCKET", min_bucket=3,
                                 mob_col="MOB", max_mob=24):
    """
    对每个开户月份的 Vintage Cohort，计算累计 Bad Rate 随 MOB 增长的曲线。
    返回 pivot table: index=vintage(开户月, 如 '2025-01'), columns=MOB, values=累计Bad率(%)
    """
    merged = df.merge(
        account_summary[[id_col, "approve_month"]], on=id_col, how="left"
    )
    merged["vintage"] = merged["approve_month"].dt.to_period("M").astype(str)

    merged = merged.sort_values([id_col, mob_col])
    merged["_is_bad_event"] = merged[bucket_col] >= min_bucket
    merged["_cum_bad"] = merged.groupby(id_col)["_is_bad_event"].cummax()

    pivot = merged.pivot_table(
        index="vintage", columns=mob_col, values="_cum_bad", aggfunc="mean"
    ) * 100
    pivot = pivot.loc[:, pivot.columns <= max_mob]
    return pivot.round(2)


def plot_vintage_curves(pivot, title="Vintage Cumulative Bad Rate", save_path=None):
    fig, ax = plt.subplots(figsize=(10, 6))
    for vintage in pivot.index:
        series = pivot.loc[vintage].dropna()
        ax.plot(series.index, series.values, marker="o", markersize=3, label=vintage)
    ax.set_xlabel("Months on Book (MOB)")
    ax.set_ylabel("Cumulative Bad Rate (%)")
    ax.set_title(title)
    ax.legend(title="Vintage", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    return fig


def find_maturity_point(pivot, marginal_threshold=0.3):
    """
    找到跨 Vintage 平均后的曲线开始"走平"的 MOB：
    当月度边际增量(Delta 累计Bad率) 连续低于 marginal_threshold(百分点)时，
    认为已接近成熟拐点 —— 可作为 Performance Window 长度的参考依据。
    """
    avg_curve = pivot.mean(axis=0, skipna=True).sort_index()
    marginal = avg_curve.diff()
    candidates = marginal[(marginal.index > 1) & (marginal.abs() < marginal_threshold)]
    mature_mob = candidates.index.min() if len(candidates) else np.nan
    return mature_mob, avg_curve, marginal


# =============================================================================
# 4. 一键跑完整流程
# =============================================================================

def run_full_analysis(df, id_col="CREDITCARDNO", snap_col="SNAPSHOT_DATE",
                       bucket_col="DELINQUENCYBUCKET", approve_col="APPROVEDATE",
                       bad_definitions=None, output_dir="."):
    df = prepare_data(df, id_col, snap_col, bucket_col, approve_col)

    print("\n========== 1. Roll Rate 分析 ==========")
    roll_rate_1m, _ = build_roll_rate_matrix(df, id_col, snap_col, bucket_col, n_periods=1)
    print("1个月滚动迁移矩阵 (%):")
    print(roll_rate_1m)
    plot_roll_rate_heatmap(roll_rate_1m, n_periods=1,
                            save_path=f"{output_dir}/roll_rate_1m.png")

    rate_30_to_90_6m, n_obs = eventual_roll_forward_rate(
        df, id_col, snap_col, bucket_col, from_bucket=2, to_bucket_min=4, within_months=6
    )
    print(f"\n一旦进入 30-59dpd(bucket=2)，未来6个月内滚到 90+dpd(bucket=4) 的概率: "
          f"{rate_30_to_90_6m:.1f}%  (基于 {n_obs} 个观察点)")

    print("\n========== 2. 候选 Bad 定义标注与对比 ==========")
    account_summary, bad_definitions = define_bad_flags(
        df, id_col, bucket_col, "MOB", bad_definitions
    )
    summary_df, overlap_df = compare_bad_definitions(account_summary, bad_definitions)
    print("\n各 Bad 定义在成熟样本口径下的表现:")
    print(summary_df.to_string(index=False))
    print("\n定义两两对比 (一致性 & 相关性):")
    print(overlap_df.to_string(index=False))

    print("\n========== 3. Vintage 分析 ==========")
    # 以 12个月窗口对应的 bad 定义 (min_bucket=3, 即 60dpd) 为例画 vintage 曲线
    example_def_name = next(iter(bad_definitions))
    min_bucket_example = bad_definitions[example_def_name]["min_bucket"]
    pivot = vintage_cumulative_bad_rate(
        df, account_summary, id_col, bucket_col, min_bucket=min_bucket_example, max_mob=18
    )
    plot_vintage_curves(
        pivot, title=f"Vintage Cumulative Bad Rate (bucket>={min_bucket_example})",
        save_path=f"{output_dir}/vintage_curves.png"
    )
    mature_mob, avg_curve, marginal = find_maturity_point(pivot)
    print(f"\n跨 Vintage 平均累计坏账率曲线开始走平的 MOB (成熟点参考): {mature_mob}")
    print("\n平均累计坏账率曲线 (按 MOB):")
    print(avg_curve.round(2).to_string())

    return {
        "df_prepared": df,
        "roll_rate_1m": roll_rate_1m,
        "account_summary": account_summary,
        "bad_definition_summary": summary_df,
        "bad_definition_overlap": overlap_df,
        "vintage_pivot": pivot,
        "maturity_mob": mature_mob,
    }


# =============================================================================
# 示例：用模拟数据跑通全流程（替换成你自己的真实数据即可）
# =============================================================================

# if __name__ == "__main__":
#     rng = np.random.RandomState(42)

#     n_accounts = 800
#     approve_dates = pd.to_datetime(
#         rng.choice(pd.date_range("2024-01-01", "2025-06-01", freq="MS"), n_accounts)
#     )
#     snapshot_end = pd.Timestamp("2026-08-01")

#     records = []
#     # 简单的马尔可夫链模拟逐月 bucket 演化: 0->1->2->3->4, 也允许部分回正常
#     trans_matrix = {
#         0: [0.90, 0.10, 0.00, 0.00, 0.00],
#         1: [0.35, 0.35, 0.30, 0.00, 0.00],
#         2: [0.10, 0.20, 0.30, 0.40, 0.00],
#         3: [0.05, 0.05, 0.10, 0.30, 0.50],
#         4: [0.02, 0.02, 0.02, 0.04, 0.90],
#     }

#     for i in range(n_accounts):
#         acc_id = f"CC{i:06d}"
#         act_date = approve_dates[i]
#         month = act_date
#         bucket = 0
#         while month <= snapshot_end:
#             records.append((acc_id, month, bucket, act_date))
#             probs = trans_matrix[bucket]
#             bucket = rng.choice([0, 1, 2, 3, 4], p=probs)
#             month = month + pd.DateOffset(months=1)

#     raw_df = pd.DataFrame(
#         records, columns=["CREDITCARDNO", "SNAPSHOT_DATE", "DELINQUENCYBUCKET", "APPROVEDATE"]
#     )

#     bad_defs = {
#         "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
#         "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
#         "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
#     }

#     results = run_full_analysis(raw_df, bad_definitions=bad_defs, output_dir="/home/claude")
